# LAB 07 — POSITIONAL ENCODING

## 1. Mục tiêu

Phần này tìm hiểu cách Transformer biểu diễn thông tin vị trí của các token trong một sequence.

Các nhiệm vụ:

- Hiểu nguyên lý Positional Encoding.
- Tính toán Sinusoidal Positional Encoding bằng tay.
- Cài đặt class `PositionalEncoding` bằng PyTorch.
- Thực nghiệm để quan sát ảnh hưởng của thông tin vị trí đến biểu diễn token.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

NumPy: 2.5.3
PyTorch: 2.14.1+cpu


In [2]:
d_model = 4
positions = [0, 1, 2]

for pos in positions:
    pe = []

    for i in range(d_model // 2):
        angle = pos / (10000 ** (2 * i / d_model))
        pe.append(np.sin(angle))
        pe.append(np.cos(angle))

    print(f"Position {pos}: {np.round(pe, 6)}")

Position 0: [0. 1. 0. 1.]
Position 1: [0.841471 0.540302 0.01     0.99995 ]
Position 2: [ 0.909297 -0.416147  0.019999  0.9998  ]


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        sequence_length = x.size(1)
        x = x + self.pe[:, :sequence_length, :]

        return x

In [4]:
d_model = 4
batch_size = 2
sequence_length = 3

embedding = torch.zeros(batch_size, sequence_length, d_model)

positional_encoding = PositionalEncoding(d_model)

output = positional_encoding(embedding)

print("Input shape:", embedding.shape)
print("Output shape:", output.shape)

print("\nPositional Encoding:")
print(positional_encoding.pe[0, :sequence_length])

print("\nOutput:")
print(output)

Input shape: torch.Size([2, 3, 4])
Output shape: torch.Size([2, 3, 4])

Positional Encoding:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998]])

Output:
tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]],

        [[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]]])


## Experiment 1 — Ảnh hưởng của thứ tự token

Sử dụng hai sequence:

- Sequence A: `I love NLP`
- Sequence B: `NLP love I`

Hai sequence chứa cùng các token nhưng có thứ tự khác nhau.

Để kiểm soát thí nghiệm, tạo một Embedding cố định cho mỗi token. Như vậy, sự khác biệt giữa hai kết quả chỉ đến từ vị trí của token.

### Các bước

1. Tạo Embedding cho ba token.
2. Đưa hai sequence vào cùng một hệ thống Embedding.
3. Quan sát biểu diễn trước Positional Encoding.
4. Cộng Positional Encoding.
5. So sánh biểu diễn sau khi cộng PE.

**Prediction:** Cùng một token sẽ có biểu diễn khác nhau khi xuất hiện ở vị trí khác nhau, do được cộng với Positional Encoding tương ứng.

In [5]:
token_embeddings = {
    "I": [0.2, 0.4, 0.6, 0.8],
    "love": [0.1, 0.3, 0.5, 0.7],
    "NLP": [0.9, 0.7, 0.5, 0.3]
}

sequence_A = ["I", "love", "NLP"]
sequence_B = ["NLP", "love", "I"]

def get_embeddings(sequence):
    return torch.tensor(
        [token_embeddings[token] for token in sequence],
        dtype=torch.float32
    ).unsqueeze(0)

embedding_A = get_embeddings(sequence_A)
embedding_B = get_embeddings(sequence_B)

output_A = positional_encoding(embedding_A)
output_B = positional_encoding(embedding_B)

print("Sequence A:", sequence_A)
print("Sequence B:", sequence_B)

print("\nEmbedding A:")
print(embedding_A)

print("\nEmbedding B:")
print(embedding_B)

print("\nAfter Positional Encoding - A:")
print(output_A)

print("\nAfter Positional Encoding - B:")
print(output_B)

Sequence A: ['I', 'love', 'NLP']
Sequence B: ['NLP', 'love', 'I']

Embedding A:
tensor([[[0.2000, 0.4000, 0.6000, 0.8000],
         [0.1000, 0.3000, 0.5000, 0.7000],
         [0.9000, 0.7000, 0.5000, 0.3000]]])

Embedding B:
tensor([[[0.9000, 0.7000, 0.5000, 0.3000],
         [0.1000, 0.3000, 0.5000, 0.7000],
         [0.2000, 0.4000, 0.6000, 0.8000]]])

After Positional Encoding - A:
tensor([[[0.2000, 1.4000, 0.6000, 1.8000],
         [0.9415, 0.8403, 0.5100, 1.6999],
         [1.8093, 0.2839, 0.5200, 1.2998]]])

After Positional Encoding - B:
tensor([[[ 0.9000,  1.7000,  0.5000,  1.3000],
         [ 0.9415,  0.8403,  0.5100,  1.6999],
         [ 1.1093, -0.0161,  0.6200,  1.7998]]])


In [6]:
print("Token I - Position 0:")
print(output_A[0, 0])

print("\nToken I - Position 2:")
print(output_B[0, 2])

print("\nAre they equal?")
print(torch.allclose(output_A[0, 0], output_B[0, 2]))

Token I - Position 0:
tensor([0.2000, 1.4000, 0.6000, 1.8000])

Token I - Position 2:
tensor([ 1.1093, -0.0161,  0.6200,  1.7998])

Are they equal?
False


In [7]:
print("Embedding I - Sequence A:")
print(embedding_A[0, 0])

print("\nEmbedding I - Sequence B:")
print(embedding_B[0, 2])

print("\nAre embeddings equal?")
print(torch.allclose(embedding_A[0, 0], embedding_B[0, 2]))

Embedding I - Sequence A:
tensor([0.2000, 0.4000, 0.6000, 0.8000])

Embedding I - Sequence B:
tensor([0.2000, 0.4000, 0.6000, 0.8000])

Are embeddings equal?
True


## Experiment 2 — Sentence Order

### Mục tiêu

Kiểm tra vai trò của thứ tự token trong việc biểu diễn câu.

So sánh hai câu:

- `dog bites man`
- `man bites dog`

Hai câu có cùng tập hợp từ nhưng khác thứ tự, dẫn đến ý nghĩa khác nhau.

### Prediction

Nếu không có Positional Encoding, Self-Attention không thể tự phân biệt thứ tự ban đầu của các token. Khi có Positional Encoding, thông tin vị trí được đưa vào biểu diễn đầu vào.

### Phương pháp

1. Tạo Embedding cố định cho từng từ.
2. Tạo hai sequence theo hai thứ tự khác nhau.
3. So sánh biểu diễn đầu vào khi có và không có Positional Encoding.
4. Phân tích kết quả.

In [8]:
word_embeddings = {
    "dog": [0.2, 0.4, 0.6, 0.8],
    "bites": [0.1, 0.3, 0.5, 0.7],
    "man": [0.9, 0.7, 0.5, 0.3]
}

sentence_A = ["dog", "bites", "man"]
sentence_B = ["man", "bites", "dog"]

def encode_sentence(sentence):
    return torch.tensor(
        [word_embeddings[word] for word in sentence],
        dtype=torch.float32
    ).unsqueeze(0)

input_A = encode_sentence(sentence_A)
input_B = encode_sentence(sentence_B)

with_pe_A = positional_encoding(input_A)
with_pe_B = positional_encoding(input_B)

print("Without Positional Encoding:")
print("Sentence A:")
print(input_A)
print("Sentence B:")
print(input_B)

print("\nWith Positional Encoding:")
print("Sentence A:")
print(with_pe_A)
print("Sentence B:")
print(with_pe_B)

Without Positional Encoding:
Sentence A:
tensor([[[0.2000, 0.4000, 0.6000, 0.8000],
         [0.1000, 0.3000, 0.5000, 0.7000],
         [0.9000, 0.7000, 0.5000, 0.3000]]])
Sentence B:
tensor([[[0.9000, 0.7000, 0.5000, 0.3000],
         [0.1000, 0.3000, 0.5000, 0.7000],
         [0.2000, 0.4000, 0.6000, 0.8000]]])

With Positional Encoding:
Sentence A:
tensor([[[0.2000, 1.4000, 0.6000, 1.8000],
         [0.9415, 0.8403, 0.5100, 1.6999],
         [1.8093, 0.2839, 0.5200, 1.2998]]])
Sentence B:
tensor([[[ 0.9000,  1.7000,  0.5000,  1.3000],
         [ 0.9415,  0.8403,  0.5100,  1.6999],
         [ 1.1093, -0.0161,  0.6200,  1.7998]]])


In [9]:
print("Inputs equal:", torch.equal(input_A, input_B))

print(
    "Mean representations equal without PE:",
    torch.allclose(input_A.mean(dim=1), input_B.mean(dim=1))
)

print(
    "Mean representations equal with PE:",
    torch.allclose(with_pe_A.mean(dim=1), with_pe_B.mean(dim=1))
)

Inputs equal: False
Mean representations equal without PE: True
Mean representations equal with PE: True


## Analysis — Experiment 2

### 1. Phân tích kết quả

Kết quả thực nghiệm:

- `Inputs equal: False`
- `Mean representations equal without PE: True`
- `Mean representations equal with PE: True`

Giải thích:

1. Hai input khác nhau vì thứ tự token trong hai câu khác nhau.
2. Mean Pooling không có Positional Encoding cho kết quả giống nhau vì hai câu chứa cùng các Embedding, chỉ khác thứ tự.
3. Khi có Positional Encoding, biểu diễn của từng token thay đổi theo vị trí. Tuy nhiên, cả hai sequence đều sử dụng cùng tập hợp vị trí PE (0, 1, 2).
4. Khi lấy trung bình toàn bộ sequence, tổng các Embedding và tổng các Positional Encoding vẫn giống nhau. Vì vậy, hai Mean Representations tiếp tục bằng nhau.

### 2. Đánh giá Prediction

Prediction được xác nhận ở cấp độ biểu diễn từng token: Positional Encoding đưa thông tin vị trí vào từng token.

Tuy nhiên, thí nghiệm cũng cho thấy Mean Pooling có thể làm mất thông tin về thứ tự khi tổng hợp sequence.

### 3. Conclusion

Positional Encoding là thành phần cần thiết để Transformer tiếp nhận thông tin vị trí. Tuy nhiên, việc giữ lại thông tin thứ tự ở biểu diễn cuối cùng còn phụ thuộc vào cách tổng hợp các token.

Đây là lý do cần nghiên cứu thêm Self-Attention và các phương pháp Pooling ở những phần tiếp theo.